# Memory Consolidation – Summary Generator
This notebook generates daily summaries for all participants and updates the vector store.
Run nightly at 1:00 AM IST.

In [ ]:
!pip install -q aiohttp python-dotenv

import asyncio
import json
import hashlib
from datetime import datetime, timedelta
import aiohttp

In [ ]:
# Load secrets from environment
import os
VAULT_API_URL = os.environ.get("VAULT_API_URL")
VAULT_API_KEY = os.environ.get("VAULT_API_KEY")
VECTORIZE_API_URL = os.environ.get("VECTORIZE_API_URL")
VECTORIZE_API_TOKEN = os.environ.get("VECTORIZE_API_TOKEN")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

print("✅ Environment loaded")

In [ ]:
async def generate_summary(session, persona):
    """Generate summary using Groq."""
    convo = "\n".join([f"{m['role']}: {m['content']}" for m in session.get("chat_messages", [])[:30]])
    if persona == "Samara":
        prompt = f"""You are Samara. Create a warm, personal summary of today's conversation with {session.get('name', 'participant')}. Note emotions and personal details.\nConversation:\n{convo}\nSummary:"""
    else:
        prompt = f"""You are Artery. Create a factual summary of today's interaction. Be neutral.\nConversation:\n{convo}\nSummary:"""
    
    async with aiohttp.ClientSession() as client:
        headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}
        payload = {"model": "llama-3.1-8b-instant", "messages": [{"role": "user", "content": prompt}], "max_tokens": 300}
        async with client.post("https://api.groq.com/openai/v1/chat/completions", headers=headers, json=payload) as resp:
            data = await resp.json()
            return data["choices"][0]["message"]["content"]

In [ ]:
async def main():
    ist_now = datetime.utcnow() + timedelta(hours=5, minutes=30)
    yesterday = (ist_now - timedelta(days=1)).strftime("%Y-%m-%d")
    print(f"Processing {yesterday}")
    # Fetch sessions and process...
    print("Done")

if __name__ == "__main__":
    await main()